# Exploratory Data Analysis — Home Credit Default Risk

**Mục tiêu:** Phân tích dữ liệu khám phá trước khi xây dựng mô hình.

**Dataset:** Home Credit Default Risk (Kaggle)
- 307,511 khách hàng
- 7 bảng dữ liệu liên kết
- Target: TARGET (0 = trả nợ đúng hạn, 1 = vỡ nợ)

In [ ]:
# Setup
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Khởi tạo Spark
spark = (
    SparkSession.builder
    .appName('eda-notebook')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

## 1. Load dữ liệu

In [ ]:
# Load bảng chính
app = spark.read.csv('../data/01_raw/application_train.csv', header=True, inferSchema=True)

print(f'Số dòng: {app.count():,}')
print(f'Số cột: {len(app.columns)}')
print()
app.printSchema()

In [ ]:
# Tổng quan 7 bảng
tables = {
    'application_train': '../data/01_raw/application_train.csv',
    'bureau': '../data/01_raw/bureau.csv',
    'bureau_balance': '../data/01_raw/bureau_balance.csv',
    'previous_application': '../data/01_raw/previous_application.csv',
    'POS_CASH_balance': '../data/01_raw/POS_CASH_balance.csv',
    'installments_payments': '../data/01_raw/installments_payments.csv',
    'credit_card_balance': '../data/01_raw/credit_card_balance.csv',
}

print(f'{"Table":<25} {"Rows":>12} {"Columns":>8}')
print('-' * 50)
for name, path in tables.items():
    if os.path.exists(path):
        df = spark.read.csv(path, header=True, inferSchema=True)
        print(f'{name:<25} {df.count():>12,} {len(df.columns):>8}')
    else:
        print(f'{name:<25} {"(not found)":>12}')

## 2. Phân bố biến mục tiêu (TARGET)

In [ ]:
# Phân bố TARGET
target_dist = app.groupBy('TARGET').count().toPandas()
target_dist['pct'] = (target_dist['count'] / target_dist['count'].sum() * 100).round(2)
print(target_dist)

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
colors = ['#27ae60', '#e74c3c']
ax.bar(['Non-default (0)', 'Default (1)'], target_dist.sort_values('TARGET')['count'], color=colors)
ax.set_ylabel('Số khách hàng')
ax.set_title(f'Phân bố TARGET — Imbalance ratio: {target_dist["pct"].min():.1f}% / {target_dist["pct"].max():.1f}%')
for i, v in enumerate(target_dist.sort_values('TARGET')['count']):
    ax.text(i, v + 2000, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Missing values

In [ ]:
# Tính % missing per column
total_rows = app.count()
null_counts = app.select([
    F.sum(F.col(c).isNull().cast('int')).alias(c) for c in app.columns
]).toPandas().T

null_counts.columns = ['null_count']
null_counts['pct_missing'] = (null_counts['null_count'] / total_rows * 100).round(2)
null_counts = null_counts.sort_values('pct_missing', ascending=False)

# Top 20 columns by missing %
top_missing = null_counts[null_counts['pct_missing'] > 0].head(20)
print(f'Columns with missing values: {(null_counts["pct_missing"] > 0).sum()} / {len(app.columns)}')
print(f'Columns with >50% missing: {(null_counts["pct_missing"] > 50).sum()}')
print()

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top_missing.index[::-1], top_missing['pct_missing'][::-1], color='#e74c3c', alpha=0.7)
ax.axvline(x=50, color='red', linestyle='--', label='Threshold 50%')
ax.set_xlabel('% Missing')
ax.set_title('Top 20 columns by missing value percentage')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Phân bố các features quan trọng

In [ ]:
# Spark describe() cho các features chính
key_features = ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 
                'DAYS_BIRTH', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

app.select(key_features).describe().toPandas()

In [ ]:
# Phân bố AMT_CREDIT theo TARGET
pdf = app.select('AMT_CREDIT', 'TARGET').toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
for target, color, label in [(0, '#27ae60', 'Non-default'), (1, '#e74c3c', 'Default')]:
    subset = pdf[pdf['TARGET'] == target]
    axes[0].hist(subset['AMT_CREDIT'], bins=50, alpha=0.6, color=color, label=label, density=True)
axes[0].set_xlabel('AMT_CREDIT')
axes[0].set_title('Phân bố số tiền vay theo TARGET')
axes[0].legend()

# Box plot
pdf.boxplot(column='AMT_CREDIT', by='TARGET', ax=axes[1])
axes[1].set_title('Box plot AMT_CREDIT theo TARGET')
axes[1].set_xlabel('TARGET')

plt.tight_layout()
plt.show()

In [ ]:
# External sources — features mạnh nhất
ext_pdf = app.select('EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'TARGET').toPandas()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, col in enumerate(['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']):
    for target, color in [(0, '#27ae60'), (1, '#e74c3c')]:
        subset = ext_pdf[ext_pdf['TARGET'] == target][col].dropna()
        axes[i].hist(subset, bins=40, alpha=0.5, color=color, label=f'TARGET={target}', density=True)
    axes[i].set_title(col)
    axes[i].legend()

plt.suptitle('External credit scores — key predictors', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Correlation matrix

In [ ]:
# Correlation matrix cho numeric features chính
corr_features = ['TARGET', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
                 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
                 'DAYS_ID_PUBLISH', 'DAYS_REGISTRATION']

corr_pdf = app.select(corr_features).toPandas()
corr_matrix = corr_pdf.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation matrix — key features vs TARGET')
plt.tight_layout()
plt.show()

# Top correlations với TARGET
target_corr = corr_matrix['TARGET'].drop('TARGET').sort_values(key=abs, ascending=False)
print('\nCorrelation với TARGET (|r| giảm dần):')
for feat, r in target_corr.items():
    direction = '↑ risk' if r > 0 else '↓ risk'
    print(f'  {feat:<25} {r:+.4f}  ({direction})')

## 6. Categorical features

In [ ]:
# Default rate theo categorical features
cat_features = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
                'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, feat in enumerate(cat_features):
    ax = axes[idx // 3][idx % 3]
    dr = (
        app.groupBy(feat)
        .agg(
            F.count('*').alias('count'),
            F.avg('TARGET').alias('default_rate'),
        )
        .orderBy('default_rate', ascending=False)
        .toPandas()
    )
    ax.barh(dr[feat], dr['default_rate'], color='#3498db', alpha=0.7)
    ax.set_xlabel('Default rate')
    ax.set_title(feat)
    ax.axvline(x=0.0807, color='red', linestyle='--', alpha=0.5, label='Overall avg')

plt.suptitle('Default rate by categorical features (red line = population average 8.07%)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Key insights

1. **Imbalanced dataset**: 91.9% non-default vs 8.1% default → cần class weight balancing
2. **External scores là features mạnh nhất**: EXT_SOURCE_2 (r = -0.16), EXT_SOURCE_3 (r = -0.18) — external credit bureaus are powerful
3. **Missing values đáng kể**: 67 columns có missing values, 41 columns >50% missing → cần imputation + drop threshold
4. **Multi-table value**: Bureau và installment history chứa thông tin bổ sung quan trọng (overdue, payment patterns)
5. **Feature engineering potential**: DTI ratio, employment tenure, credit utilization → domain features cải thiện model

In [ ]:
spark.stop()
print('Done!')